# Invoice-Level Collections Prediction

This Colab notebook demonstrates a simple treasury collections workflow using synthetic invoice data. It focuses on invoice-level late-payment prediction, prioritization, and DSO-oriented interpretation.

## Learning goals

- Build invoice-level features from payment history, terms, disputes, relationship quality, and seasonality.
- Train a lightweight late-payment classifier.
- Rank invoices for collections attention.
- Estimate how better collections timing supports working-capital improvement.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

rng = np.random.default_rng(42)

In [ ]:
n = 1200
industry = rng.choice(['Technology', 'Retail', 'Manufacturing', 'Healthcare'], size=n, p=[0.3, 0.25, 0.25, 0.2])
region = rng.choice(['North', 'South', 'East', 'West'], size=n)
channel = rng.choice(['Email', 'Portal', 'Distributor', 'Field Sales'], size=n, p=[0.25, 0.3, 0.2, 0.25])
payment_terms = rng.choice([15, 30, 45, 60], size=n, p=[0.15, 0.45, 0.25, 0.15])
invoice_amount = np.round(rng.lognormal(mean=13.2, sigma=0.7, size=n) / 1e5, 2)
avg_days_beyond_terms = np.clip(rng.normal(12, 9, size=n), 0, 60)
payment_history_score = np.clip(100 - avg_days_beyond_terms * 1.7 + rng.normal(0, 8, size=n), 25, 98)
open_dispute = rng.binomial(1, 0.14, size=n)
relationship_strength = rng.choice(['Weak', 'Stable', 'Strategic'], size=n, p=[0.2, 0.55, 0.25])
seasonality_stress = rng.choice(['Low', 'Moderate', 'High'], size=n, p=[0.45, 0.35, 0.2])
industry_stress = pd.Series(industry).map({'Technology': 0.10, 'Retail': 0.18, 'Manufacturing': 0.14, 'Healthcare': 0.08}).to_numpy()
seasonality_weight = pd.Series(seasonality_stress).map({'Low': 0.0, 'Moderate': 0.08, 'High': 0.16}).to_numpy()
relationship_weight = pd.Series(relationship_strength).map({'Weak': 0.18, 'Stable': 0.06, 'Strategic': -0.03}).to_numpy()
channel_weight = pd.Series(channel).map({'Email': 0.04, 'Portal': 0.01, 'Distributor': 0.09, 'Field Sales': -0.02}).to_numpy()

risk_signal = (
    0.025 * avg_days_beyond_terms
    - 0.018 * (payment_history_score / 10)
    + 0.22 * open_dispute
    + industry_stress
    + seasonality_weight
    + relationship_weight
    + channel_weight
    + rng.normal(0, 0.08, size=n)
)

late_payment = (risk_signal > np.quantile(risk_signal, 0.56)).astype(int)

df = pd.DataFrame({
    'industry': industry,
    'region': region,
    'channel': channel,
    'payment_terms': payment_terms,
    'invoice_amount_lakh': invoice_amount,
    'avg_days_beyond_terms': np.round(avg_days_beyond_terms, 1),
    'payment_history_score': np.round(payment_history_score, 1),
    'open_dispute': open_dispute,
    'relationship_strength': relationship_strength,
    'seasonality_stress': seasonality_stress,
    'late_payment': late_payment
})

df.head()

In [ ]:
target = 'late_payment'
numeric_features = ['payment_terms', 'invoice_amount_lakh', 'avg_days_beyond_terms', 'payment_history_score', 'open_dispute']
categorical_features = ['industry', 'region', 'channel', 'relationship_strength', 'seasonality_stress']

X = df[numeric_features + categorical_features]
y = df[target]

numeric_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_transformer = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

model = Pipeline([
    ('prep', preprocessor),
    ('clf', RandomForestClassifier(n_estimators=200, max_depth=8, random_state=42))
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
model.fit(X_train, y_train)
pred = model.predict(X_test)
proba = model.predict_proba(X_test)[:, 1]

print('ROC AUC:', round(roc_auc_score(y_test, proba), 3))
print(classification_report(y_test, pred))

In [ ]:
scored = df.copy()
scored['late_risk_probability'] = model.predict_proba(X)[:, 1]
scored['cash_at_risk_lakh'] = scored['invoice_amount_lakh'] * scored['late_risk_probability']
scored['priority_band'] = pd.cut(scored['late_risk_probability'], bins=[0, 0.35, 0.6, 1], labels=['Monitor', 'Targeted follow-up', 'Immediate action'])

priority_view = scored.sort_values(['late_risk_probability', 'invoice_amount_lakh'], ascending=[False, False]).head(15)
priority_view[['industry', 'region', 'channel', 'invoice_amount_lakh', 'avg_days_beyond_terms', 'payment_history_score', 'open_dispute', 'relationship_strength', 'late_risk_probability', 'priority_band']]

In [ ]:
annual_invoiced_lakh = scored['invoice_amount_lakh'].sum()
cash_freed_per_dso_day_lakh = annual_invoiced_lakh / 365
high_risk_share = scored.loc[scored['priority_band'] == 'Immediate action', 'invoice_amount_lakh'].sum() / annual_invoiced_lakh

summary = pd.DataFrame({
    'metric': [
        'Total synthetic invoice value (lakh)',
        'Cash freed if DSO improves by 1 day (lakh)',
        'Share of value in immediate-action band'
    ],
    'value': [
        round(annual_invoiced_lakh, 2),
        round(cash_freed_per_dso_day_lakh, 2),
        round(high_risk_share * 100, 2)
    ]
})

summary

## Discussion prompts

- Which features look operationally actionable versus merely descriptive?
- When should a strategic relationship lower collections urgency even if a model predicts delay?
- How would you connect the priority table to DSO improvement and working-capital release targets?